# AGENTIC AI
## SESSION 18

Assignment questions and answers


## Question 1

Set up a simple MCP (Multi-Client Protocol) server script that listens on a specified port and prints a message whenever a client connects.

**Answer:**

```python
import asyncio
from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("SimpleServer", port=8000)


# Custom middleware/lifespan to log connections
@mcp.tool()
def ping() -> str:
    """A simple tool to test connection."""
    return "pong"


if __name__ == "__main__":
    print("Server running on port 8000...")
    # Run using SSE (Server-Sent Events) transport for network port listening
    mcp.run(transport="sse")
```

*(Note: When a client connects via the `/sse` endpoint, the underlying ASGI server automatically prints a message: `INFO: ... - "GET /sse HTTP/1.1" 200 OK`)*

## Question 2

Modify your MCP server to handle file listing requests: when a client sends the command 'LIST', respond with the names of all files in a given directory (for example, a 'music' folder containing MP3 files, like a mini Spotify backend).

**Answer:**

```python
import os
from mcp.server.fastmcp import FastMCP

# Initialize the FastMCP server
mcp = FastMCP("MiniSpotifyServer")

MUSIC_DIR = "music"


@mcp.tool()
def handle_command(command: str) -> list[str]:
    """Handle commands. Pass 'LIST' to list all music files."""
    if command.strip().upper() == "LIST":
        if not os.path.exists(MUSIC_DIR):
            return ["Error: Directory not found"]
        # Return all files ending with .mp3
        return [f for f in os.listdir(MUSIC_DIR) if f.endswith(".mp3")]
    return ["Unknown command"]


if __name__ == "__main__":
    mcp.run()
```

## Question 3

Extend your MCP server to support file download: when a client sends 'GET filename', send the contents of that file back to the client. Test this by downloading a sample image or text file from your server.

**Answer:**

Here is the extended MCP server implementation supporting file download:

```python
import base64
import os
from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("FileServer")


@mcp.tool()
def get_file(command: str) -> dict:
    """Handles 'GET filename' and returns the file content."""
    parts = command.strip().split(maxsplit=1)

    if len(parts) != 2 or parts[0].upper() != "GET":
        return {"error": "Invalid format. Use: 'GET filename'"}

    filename = parts[1]

    if not os.path.exists(filename):
        return {"error": f"File '{filename}' not found."}

    try:
        # Read file as binary to handle both text and images
        with open(filename, "rb") as f:
            content = f.read()

        # Try decoding as text; fallback to base64 for binary files (e.g., images)
        try:
            return {
                "filename": filename,
                "type": "text",
                "content": content.decode("utf-8"),
            }
        except UnicodeDecodeError:
            return {
                "filename": filename,
                "type": "binary",
                "content": base64.b64encode(content).decode("utf-8"),
            }
    except Exception as e:
        return {"error": str(e)}


if __name__ == "__main__":
    # Test script: Create a sample text file and verify download
    test_filename = "sample.txt"
    with open(test_filename, "w") as f:
        f.write("Hello, this is a sample file downloaded from the MCP server.")

    # Execute test request
    response = get_file(f"GET {test_filename}")
    print("Test Response:", response)
```

## Question 4

Add error handling to your MCP server so that if a client requests a file that does not exist, the server responds with a clear error message instead of crashing.

Hint: Use try-except (or try-catch) blocks to catch file-not-found errors and send a custom error response to the client.

**Answer:**

```python
@server.call_tool()
async def read_file(name: str, arguments: dict) -> list[types.TextContent]:
    file_path = arguments.get("path")
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        return [types.TextContent(type="text", text=content)]
    except FileNotFoundError:
        return [
            types.TextContent(
                type="text", text=f"Error: The file '{file_path}' does not exist."
            )
        ]
    except Exception as e:
        return [
            types.TextContent(type="text", text=f"Error reading file: {str(e)}")
        ]
```

## Question 5

Use ChatGPT or Copilot to generate code snippets for adding a 'DELETE filename' command to your MCP server, allowing clients to delete files from the server directory. Integrate and test this feature, and note any improvements you made to the AI-generated code.

**Answer:**

### 1. Python Code Implementation

```python
import os
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("File-Manager")
ALLOWED_DIR = os.path.abspath("./server_files")


@mcp.tool()
def delete_file(filename: str) -> str:
    """Deletes a file safely from the server directory."""
    # Prevent path traversal attacks
    target_path = os.path.abspath(os.path.join(ALLOWED_DIR, filename))
    if not target_path.startswith(ALLOWED_DIR):
        return "Error: Access denied (path traversal detected)."

    if not os.path.exists(target_path):
        return f"Error: File '{filename}' not found."

    if os.path.isdir(target_path):
        return "Error: Target is a directory, not a file."

    try:
        os.remove(target_path)
        return f"Success: File '{filename}' deleted."
    except Exception as e:
        return f"Error: Failed to delete file ({str(e)})."
```

---

### 2. Integration and Testing

* **Integration:** Registered the `delete_file` function as an MCP tool using the FastMCP framework and created the `server_files/` base directory.
* **Normal Test:** Created `test.txt` in the directory, invoked `delete_file("test.txt")`, verified successful deletion message, and confirmed file absence on disk.
* **Non-Existent File Test:** Invoked `delete_file("missing.txt")`, verified graceful return of the file-not-found error.
* **Path Traversal Test:** Tested inputs like `../../etc/passwd`, verified the server correctly blocked the request.

---

### 3. Improvements Made to AI-Generated Code

* **Path Traversal Prevention:** Replaced standard `os.remove(filename)` with `os.path.abspath` checks to prevent clients from deleting files outside `ALLOWED_DIR`.
* **Directory Check:** Added `os.path.isdir()` check to prevent errors when trying to remove directories with `os.remove`.
* **String Returns Instead of Raw Exceptions:** Caught runtime exceptions and returned meaningful error messages to prevent the MCP server connection from dropping.